# SpendDNA – Your Wallet's Year-End Story

## Student Details
Name: Vinisha
Batch: june 2026__
Project: SpendDNA – Week 2 Minor Project
Date: July 2026

### Objective
Build a fintech-style transaction analytics system that analyzes
6 months of UPI/bank transactions, performs merchant
normalization, spending categorization, anomaly detection,
behavioral archetype analysis, and generates a formatted
financial report.

In [28]:
# Libraries,LoadDataSet
import pandas as pd
import numpy as np
df = pd.read_csv("/content/Data set for DADS June.csv")

print(df.head())
print(df.shape)

          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  
(1328, 8)


In [29]:
# Remove Duplicates
duplicates = df.duplicated().sum()

df = df.drop_duplicates()

print("Duplicates removed:", duplicates)
print("Remaining rows:", len(df))

Duplicates removed: 18
Remaining rows: 1310


In [30]:
#Parse Dates
df['date'] = pd.to_datetime(
    df['Date'],
    errors='coerce',
    dayfirst=True
)

print(df['date'].head())
print("Invalid dates:", df['date'].isna().sum())

0   2024-01-01
1          NaT
2          NaT
3   2024-01-01
4          NaT
Name: date, dtype: datetime64[ns]
Invalid dates: 1167


In [31]:
#Clean Amount Column
df['amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹','',regex=False)
    .str.replace('Rs.','',regex=False)
    .str.replace(',','',regex=False)
    .str.strip()
)

df['amount'] = pd.to_numeric(
    df['amount'],
    errors='coerce'
)

print(df['amount'].head())
print("Invalid amounts:", df['amount'].isna().sum())

0     2462.0
1       50.0
2    84728.0
3     1828.0
4      270.0
Name: amount, dtype: float64
Invalid amounts: 0


In [32]:
#Standardize Transaction Type)
type_map = {
    'DR':'debit',
    'Debit':'debit',
    'CR':'credit',
    'Credit':'credit'
}

df['type_clean'] = df['Type'].map(type_map)

print(df['type_clean'].value_counts())

type_clean
debit     1304
credit       6
Name: count, dtype: int64


In [33]:
#Vendor Dictionary)
vendor_dict = {

    'Swiggy':['SWIGGY','BUNDL'],
    'Zomato':['ZOMATO'],

    'Zepto':['ZEPTO'],
    'Blinkit':['BLINKIT'],
    'Instamart':['INSTAMART'],

    'Amazon':['AMAZON','AMZN'],
    'Flipkart':['FLIPKART'],
    'Myntra':['MYNTRA'],
    'Ajio':['AJIO'],

    'Zerodha':['ZERODHA','COIN'],
    'Groww':['GROWW','NEXTBILLION'],

    'Uber':['UBER'],
    'Ola':['OLA'],
    'Rapido':['RAPIDO'],

    'Starbucks':['STARBUCKS'],
    'Third Wave':['THIRDWAVE'],
    'Cafe Coffee Day':['COFFEE DAY'],

    'Netflix':['NETFLIX'],
    'Spotify':['SPOTIFY'],

    'BookMyShow':['BOOKMYSHOW'],

    'DMart':['DMART','AVENUE'],

    'Indian Oil':['INDIAN OIL'],
    'HP':['HPCL'],
    'BPCL':['BPCL']
}

In [34]:
#Vendor Extraction Function)
def extract_vendor(desc):

    desc = str(desc).upper()

    if 'ATM' in desc:
        return 'Cash Withdrawal'

    if '@OK' in desc and 'UPI' in desc:
        return 'P2P Transfer'

    for vendor in vendor_dict:

        for keyword in vendor_dict[vendor]:

            if keyword in desc:
                return vendor

    return 'Uncategorised'

In [35]:
#Apply Vendor Extraction
df['vendor_clean'] = (
    df['Description']
    .apply(extract_vendor)
)

print(df['vendor_clean'].value_counts().head(20))
print("Unique vendors:",df['vendor_clean'].nunique())

vendor_clean
Uncategorised      333
Swiggy             181
P2P Transfer       118
Zomato             107
Amazon              86
Uber                71
Ola                 69
Zepto               58
Flipkart            43
Starbucks           42
Blinkit             40
Instamart           20
Myntra              20
Cash Withdrawal     17
Cafe Coffee Day     17
Rapido              17
Zerodha             14
DMart               13
Indian Oil          11
Netflix             10
Name: count, dtype: int64
Unique vendors: 24


In [36]:
#Category Mapping
category_map = {

    'Swiggy':'Food Delivery',
    'Zomato':'Food Delivery',

    'Zepto':'Quick Commerce',
    'Blinkit':'Quick Commerce',
    'Instamart':'Quick Commerce',

    'Amazon':'E-commerce',
    'Flipkart':'E-commerce',
    'Myntra':'E-commerce',
    'Ajio':'E-commerce',

    'Zerodha':'Investments',
    'Groww':'Investments',

    'Uber':'Transport',
    'Ola':'Transport',
    'Rapido':'Transport',

    'Starbucks':'Cafe',
    'Third Wave':'Cafe',
    'Cafe Coffee Day':'Cafe',

    'Netflix':'Subscriptions',
    'Spotify':'Subscriptions',

    'BookMyShow':'Entertainment',

    'DMart':'Groceries',

    'Indian Oil':'Fuel',
    'HP':'Fuel',
    'BPCL':'Fuel',

    'P2P Transfer':'Personal Transfer',
    'Cash Withdrawal':'Cash Withdrawal'
}

In [37]:
#Apply Category
df['category'] = (
    df['vendor_clean']
    .map(category_map)
    .fillna('Uncategorised')
)

print(df['category'].value_counts())

category
Uncategorised        333
Food Delivery        288
Transport            157
E-commerce           149
Personal Transfer    118
Quick Commerce       118
Cafe                  59
Investments           23
Subscriptions         18
Cash Withdrawal       17
Groceries             13
Fuel                  13
Entertainment          4
Name: count, dtype: int64


In [38]:
#Executive Summary
credits = df[
    df['type_clean']=='credit'
]['amount'].sum()

debits = df[
    df['type_clean']=='debit'
]['amount'].sum()

net_change = credits - debits

savings_rate = (
    net_change/credits
)*100

print("Credits:",credits)
print("Debits:",debits)
print("Net:",net_change)
print("Savings Rate:",round(savings_rate,2))

Credits: 509774.0
Debits: 1678901.0
Net: -1169127.0
Savings Rate: -229.34


In [39]:
#Top Categories
top_categories = (
    df[df['type_clean']=='debit']
    .groupby('category')['amount']
    .sum()
    .sort_values(ascending=False)
)

print(top_categories.head())

category
E-commerce           568804.0
Uncategorised        407386.0
Investments          248160.0
Food Delivery        127602.0
Personal Transfer     69196.0
Name: amount, dtype: float64


In [40]:
#Top Vendors
top_vendors = (
    df.groupby('vendor_clean')
    .agg(
        spend=('amount','sum'),
        orders=('amount','count')
    )
    .sort_values(
        'spend',
        ascending=False
    )
)

print(top_vendors.head())

                  spend  orders
vendor_clean                   
Uncategorised  917160.0     333
Amazon         328530.0      86
Zerodha        210000.0      14
Flipkart       170745.0      43
Swiggy          77799.0     181


In [41]:
#Monthly Trend
df['month'] = df['date'].dt.month

monthly = df.pivot_table(
    values='amount',
    index='category',
    columns='month',
    aggfunc='sum'
)

print(monthly)

month                  1.0     2.0      3.0     4.0      5.0     6.0   \
category                                                                
Cafe                    NaN   355.0      NaN   688.0      NaN     NaN   
Cash Withdrawal         NaN     NaN      NaN     NaN      NaN     NaN   
E-commerce           5346.0   996.0   2147.0  3311.0      NaN  3000.0   
Entertainment           NaN     NaN      NaN   343.0      NaN     NaN   
Food Delivery        1180.0  3454.0   2411.0     NaN   1159.0  1636.0   
Groceries               NaN     NaN      NaN     NaN      NaN     NaN   
Investments             NaN     NaN      NaN     NaN      NaN     NaN   
Personal Transfer    1828.0     NaN   1664.0   632.0      NaN  1860.0   
Quick Commerce       1616.0   697.0      NaN     NaN    502.0     NaN   
Subscriptions           NaN     NaN      NaN     NaN      NaN     NaN   
Transport             826.0   621.0    439.0     NaN    431.0   134.0   
Uncategorised      174270.0   267.0  19917.0   272.

In [42]:
#Time Analysis)
df['hour'] = (
    df['Time']
    .str[:2]
    .astype(int)
)

time_pattern = df.pivot_table(
    values='amount',
    index='category',
    columns='hour',
    aggfunc='count',
    fill_value=0
)

print(time_pattern)

hour               0   1   2   3   4   5   6   7   8   9   ...  14  15  16  \
category                                                   ...               
Cafe                1   2   0   1   0   0   1   2   3   3  ...   2   1   5   
Cash Withdrawal     0   0   0   0   0   0   0   0   2   3  ...   2   0   1   
E-commerce          9   3   3   8   8  10   4   3   2   9  ...   9   3   8   
Entertainment       1   0   0   0   0   0   0   0   0   0  ...   0   0   1   
Food Delivery       2   8   1   3   4   6   1   2   9  10  ...   8  14   9   
Fuel                2   0   0   1   1   1   0   0   0   0  ...   1   1   1   
Groceries           2   1   1   1   0   0   1   0   1   0  ...   0   0   0   
Investments         1   1   0   0   2   1   4   0   0   1  ...   0   0   0   
Personal Transfer   0   0   2   3   4   1   2   2   1   7  ...   8   6   4   
Quick Commerce      3   2   2   2   1   3   0   1   2  10  ...   6   6   6   
Subscriptions       0   0   1   2   5   2   3   1   0   0  ...  

In [43]:
#Anomaly Detection
mean = (
    df.groupby('category')
    ['amount']
    .transform('mean')
)

std = (
    df.groupby('category')
    ['amount']
    .transform('std')
)

df['z_score'] = (
    df['amount']-mean
)/std

anomalies = df[
    df['z_score']>2
]

print(
    anomalies[
        ['date',
         'vendor_clean',
         'amount',
         'z_score']
    ]
    .sort_values(
        'z_score',
        ascending=False
    )
    .head(10)
)

           date   vendor_clean   amount   z_score
1113        NaT  Uncategorised  85492.0  7.230454
880  2024-01-05  Uncategorised  85393.0  7.221803
658         NaT  Uncategorised  84736.0  7.164388
2           NaT  Uncategorised  84728.0  7.163688
224         NaT  Uncategorised  84724.0  7.163339
438  2024-01-03  Uncategorised  84701.0  7.161329
1298        NaT         Amazon  22008.0  3.803781
269         NaT         Amazon  21986.0  3.799180
216         NaT   P2P Transfer   2774.0  3.374231
475         NaT         Amazon  19917.0  3.366536


In [44]:
#Archetype Detection
archetypes = []

print(archetypes)

[]


In [45]:
#Final Report
print("="*60)
print("SPENDDNA REPORT")
print("="*60)


SPENDDNA REPORT


# Reflection

This project helped me understand:

- Financial data cleaning
- Vendor normalization
- Transaction analytics
- Anomaly detection
- Spending behavior analysis
- Fintech analytics workflows